In [0]:
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.functions import *
from dataclasses import dataclass

In [0]:
%sql
    
-- created table schema for store the config
create or replace table netflix.config_table (
    pipeline_name string
    , file_path string
    , header string
    , delimiter string
    , table_name string
    , schema_detail map<string,string>
    , keys array<string>
    , write_mode string
)

In [0]:
def upsert_into(df:DataFrame,table_name:str, keys:list) -> DataFrame:
    delta_obj = DeltaTable.forName(spark,table_name)
    return (
        delta_obj.alias("t").merge(
            df.alias("s")," AND ".join([f"t.{key} = s.{key}" for key in keys])
            )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
        )

In [0]:
# Helper function to check type of file
def is_valid_format_path(file_path:str) -> str:
    if file_path.split(".")[-1] == "csv":
        return True
    else:
        raise Exception(f"Invalid file format: {file_path}")

In [0]:
# Created class that use to register new data to config_table
@dataclass
class RegisterConfig:
    pipeline_name:str
    file_path:str
    header:str
    delimiter:str
    table_name:str
    schema_detail:dict
    keys:list
    write_mode:str

    def __post_init__(self) -> None:
        is_valid_format_path(self.file_path)
        self.register_table_name = "netflix.config_table"
        self.data =[{
            "pipeline_name": self.pipeline_name,
            "file_path": self.file_path,
            "header": self.header,
            "delimiter": self.delimiter,
            "table_name": self.table_name,
            "schema_detail": self.schema_detail,
            "keys": self.keys,
            "write_mode": self.write_mode
        }]

        self.schema = StructType([
            StructField("pipeline_name", StringType(), True),
            StructField("file_path", StringType(), True),
            StructField("header", StringType(), True),
            StructField("delimiter", StringType(), True),
            StructField("table_name", StringType(), True),
            StructField("schema_detail", MapType(StringType(), StringType()), True),
            StructField("keys", ArrayType(StringType()), True),
            StructField("write_mode", StringType(), True)
        ])

    # Function use to register new table to config_table in another way 
    @classmethod
    def for_netflix_config(cls, pipeline_name:str, file_name:str, table_name:str, schema_detail:dict[str, str], keys:list[str],) -> 'RegisterConfig':
        return cls(
            pipeline_name = pipeline_name
            , file_path = f"dbfs:/Volumes/workspace/netflix/raw_data/{file_name}"
            , header = "true"
            , delimiter = ","
            , table_name = table_name
            , schema_detail = schema_detail
            , keys = keys 
            , write_mode = write_mode # Use append when implement reality time
        )
    # function use for create new data to table brfore merge    
    def register_df(self) -> DataFrame:
        return spark.createDataFrame(self.data, self.schema)
    
    # function use to merge register DataFrame to config_table
    def merge_to_config_table(self, register_df: DataFrame) -> None:
        upsert_into(register_df, self.register_table_name, ["pipeline_name"])

In [0]:
pipeline_name = "netflix"
file_path = "dbfs:/Volumes/workspace/netflix/raw_data/netflix_titles.csv"
header = "true"
delimiter = ","
table_name = "workspace.netflix.netflix"
schema_detail = {
    'show_id': "string"
    , 'type': "string"
    , 'title': "string"
    , 'director': "string"
    , 'cast': "string"
    , 'country': "string"
    , 'date_added': "date"
    , 'release_year': "integer"
    , 'rating': "string"
    , 'duration': "string"
    , 'listed_in': "string"
    , 'description': "string"
}
keys = ['show_id']
write_mode = "overwrite"


In [0]:
# assign by normal method
r = RegisterConfig(
    pipeline_name = pipeline_name
    , file_path = file_path
    , header = header
    , delimiter = delimiter
    , table_name = table_name
    , schema_detail = schema_detail
    , keys = keys
    , write_mode = write_mode
)
register_df = r.register_df()
r.merge_to_config_table(register_df)


In [0]:
# %sql
# truncate table netflix.config_table

In [0]:
spark.table("workspace.netflix.config_table").display()

In [0]:
# df = (spark.read
#     .format("csv")
#     .option("header", "true")
#     .option("delimiter", ",")
#     .option("inferSchema", "true")
#     .load("/Volumes/workspace/netflix/raw_data/netflix_titles.csv")    
# )

# print(df.columns)
# print(df.schema)
# df.limit(3).display()